# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Plan: inspect data, check for leakage, build a logistic baseline and gradient boosting with a **time-based split**, evaluate with ROC-AUC / PR-AUC, and look at feature importance.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.inspection import permutation_importance
df=pd.read_csv('data.csv',parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, df.isna().sum().sum(),'missing;', df.asset_tag.nunique(),'unique assets;', df.inspected_at.min(),'->',df.inspected_at.max())
df.describe().T

Each asset appears once, so there is no asset-level leakage across rows. Base rate by quarter:

In [ ]:
print('overall', df.failed_90d.mean().round(3))
df.groupby(df.inspected_at.dt.to_period('Q')).failed_90d.agg(['mean','size'])

## Leakage check: `parts_replaced`
The dictionary says it is the parts replaced on the work order *that closed this inspection record* — recorded after the inspection, plausibly as a consequence of a failure/repair.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean','size'])

Zero parts → never fails; ≥2 parts → always fails. That near-perfect separation is information from after the outcome, not available at inspection time. **We exclude it**; a model using it would look excellent but be useless for real prediction. (Worth confirming with the data owner when this field is populated.)

In [ ]:
print('AUC using parts_replaced alone:', roc_auc_score(df.failed_90d, df.parts_replaced))
feats=['vibration_rms','bearing_temp_c','ambient_temp_c','meter_hours','load_pct']
fig,ax=plt.subplots(1,5,figsize=(18,3))
for a,f in zip(ax,feats):
    for v in (0,1): a.hist(df.loc[df.failed_90d==v,f],bins=40,alpha=.55,density=True,label=str(v))
    a.set_title(f)
ax[0].legend(title='failed_90d'); plt.tight_layout(); plt.show()
print(df.groupby('site').failed_90d.mean().round(3).to_dict())
df[feats+['failed_90d']].corr()['failed_90d']

## Modelling (without `parts_replaced`)
Train on the first 75% of inspections by time, test on the most recent 25%.

In [ ]:
df['temp_rise']=df.bearing_temp_c-df.ambient_temp_c
X_cols=feats+['temp_rise','site']
X,y=df[X_cols],df.failed_90d
cut=int(len(df)*.75)
Xtr,Xte,ytr,yte=X[:cut],X[cut:],y[:cut],y[cut:]
print('train rate',ytr.mean().round(3),'test rate',yte.mean().round(3))
num=[c for c in X_cols if c!='site']
lr=make_pipeline(ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(),['site'])]),LogisticRegression(max_iter=1000))
gb=make_pipeline(ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(),['site'])]),
                 HistGradientBoostingClassifier(max_depth=3,learning_rate=.05,max_iter=300,l2_regularization=1.0,random_state=0))
res={}
for n,m in [('logreg',lr),('hgb',gb)]:
    m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; res[n]=p
    cv=cross_val_score(m,Xtr,ytr,cv=TimeSeriesSplit(5),scoring='roc_auc')
    print(f'{n}: test ROC-AUC {roc_auc_score(yte,p):.3f}  PR-AUC {average_precision_score(yte,p):.3f}  (base {yte.mean():.3f})  | time-CV AUC {cv.mean():.3f}+-{cv.std():.3f}')

In [ ]:
best='hgb' if roc_auc_score(yte,res['hgb'])>=roc_auc_score(yte,res['logreg']) else 'logreg'
p=res[best]; print('best:',best)
pr,rc,th=precision_recall_curve(yte,p)
plt.plot(rc,pr);plt.axhline(yte.mean(),ls='--',c='gray');plt.xlabel('recall');plt.ylabel('precision');plt.title('Precision-recall (test)');plt.show()
d=pd.DataFrame({'p':p,'y':yte.values}); d['decile']=pd.qcut(d.p.rank(method='first'),10,labels=False)
d.groupby('decile').y.agg(['mean','size']).round(3).T

In [ ]:
m={'hgb':gb,'logreg':lr}[best]
r=permutation_importance(m,Xte,yte,scoring='roc_auc',n_repeats=20,random_state=0)
pd.Series(r.importances_mean,index=X_cols).sort_values().plot.barh(title='Permutation importance (drop in AUC)');plt.show()

## Conclusions
* **Leakage:** `parts_replaced` predicts the target almost perfectly (AUC 0.997 alone) but is recorded on the work order closing the record, i.e. after the outcome. It is excluded. Any "very accurate" model built with it should not be trusted.
* **Real signal is modest.** Without it, gradient boosting gets test ROC-AUC ~0.75 (time-CV ~0.76) and logistic regression ~0.66-0.72. Test PR-AUC ~0.11 vs a 4.3% base rate: failures are ranked better than chance (top risk decile fails ~13% vs <1% in the bottom two), but precision is low, so this suits prioritising inspections, not confident individual calls.
* **Drivers:** load_pct and vibration_rms correlate positively with failure; meter_hours negatively. Permutation importance on the test period is dominated by load_pct; the rest are small/noisy. Site carries no signal.
* **Caveats:** the failure rate is far from stable over time (~4% in 2024Q1, ~27% in 2024Q2-Q3, ~3% in 2025Q4), so train/test base rates differ (14% vs 4%) and calibrated probabilities will not transfer. The low late-2025 rate may partly be right-censoring if failures after the data extract are not yet observed - worth checking with the data owner. Each asset appears only once, so per-asset history (trends in vibration) cannot be used.
* **Next steps:** confirm label construction/censoring, obtain repeated inspections per asset to build trend features, and tune the decision threshold to inspection capacity.